### Connectivity Based Clustering of DN/AN types
This notebook applies some basic embedding (UMAP), factorization (PCA, NMF), and clustering (spectral clustering) algorithms to descending neuron input/output connectivity. Each DN type (e.g. DNa02, DNg12) is defined as a unit vector (i.e. direction) in a high dimensional connectivity space whose axes correspond to lateralized connections with cell types in the VNC (e.g. IN02A032_contralateral, vPR9_inward, w-cHIN_contralateral). This representation captures each DN  type's pattern of connectivity in a manner that is invariant to its side (left, right, center) and its number of outputs (since each vector is normalized). Input connectivity is defined analogously. Throughout this notebook, cosine similarity (which always ranges between 0 and 1 in our case, since the vectors are non-negative) is used to quantify closeness in connectivity space. This metric is also used in the mushroom body connectome paper to cluster MBONs.

On the output side, we find 6 DN output clusters: an abdominal cluster, ipsi and contralateral "wing" (or in any case, dorsal VNC) clusters, and contralateral, ipsilateral, and nonlateral leg clusters. A finer clustering of outputs is probably possible, though less obvious. The analysis of DN input connectivity also reveals some interesting, though less striking patterns. For example, the leg, wing, and abdominal DNs identified by the output clustering define largely contiguous regions of the input connectivity space. There also seems to be a gradient among the leg DNs regarding degree of lateralization of outputs in the VNC, though in 2D this does not seem to separate ipsi and contralateral, which is interesting. There are also suggestions that this lateralized part of the connectivity space contains a lot of turning related DNs and is correspondingly enriched in brain inputs associated with turning.

Once AN connectivity is added to the frankenbrain, we will apply the same methods to ascending neuron connectivity analysis.

Practically speaking, the main output of this notebook is a huge number of plots overlayed with useful meta data. As more annotations are added to the dataset, it should be possible to overlay these, as the code is written in a fairly modular and extendible way.

Some notes: we do not consider connectivity from descending/ascending neurons to other descending/ascending neurons in this analysis. This could, and perhaps should, be changed. We also apply a thresholding where we throw out any connectivity to cell types that do not account for at least 5% of a cell types inputs or outputs. This doesn't seem to have a big effect either way, and in any case, it's a tunable parameter. Finally, this code is written such that it should produce the same outputs every time the notebook is run, since random seeds are specified for each non-deterministic algorithm.

In [ ]:
import numpy as np
import pandas as pd
import sqlite3
import random

import matplotlib.pyplot as plt
from matplotlib import cm

In [ ]:
seed = 0
random.seed(seed)
np.random.seed(seed)

In [ ]:
del_input = False
del_output = False

## Setup

In [ ]:
sql_path = 'frankenbrain_v.1.0_data.sqlite'
index_col = 'id'

# Connect to the SQLite database
conn = sqlite3.connect(sql_path)

cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

# Print the tables
print("Tables in the database:", tables)

query = f"SELECT * FROM meta"
all_meta = pd.read_sql_query(query, conn)
conn.close()

all_meta = all_meta[all_meta['id'] != '']
all_meta[index_col] = all_meta[index_col].values.astype(int)
all_meta = all_meta.set_index(index_col, drop=False)
all_meta = all_meta.replace('', np.nan)
all_meta = all_meta[all_meta['side'] != 'na']
all_meta['side'] = all_meta['side'].replace('midline', 'center')
all_meta = all_meta[~all_meta.index.duplicated(keep='first')]

indices = all_meta.index.values
vnc_mask = all_meta['root_id'].isnull()
all_meta['is_vnc'] = vnc_mask

all_meta = all_meta[~all_meta['cell_type'].isnull()]

all_meta.loc[all_meta['cell_class'] == 'ascending_neuron', 'super_class'] = 'ascending'
all_meta.loc[all_meta['cell_class'] == 'descending_neuron', 'super_class'] = 'descending'
all_meta.loc[all_meta['cell_class'] == 'sensory_ascending', 'super_class'] = 'sensory_ascending'

In [ ]:
# We want to discard left right info in the origin field and simplify entries that span > 2 neuropils
# by somewhat arbitrarily taking the first. We will also simply ignore multi as a label.

origins = []
for val in all_meta['origin'].values:
    if type(val) == float:
        pass
    elif val == 'multi':
        val = np.nan
    else:
        val = val.replace('_L', '').replace('_R', '').replace('tct', 'Tct')
        val = val.split('.')[0]
        
    origins.append(val)
all_meta['origin'] = origins

In [ ]:
coarse_origin = []
for val in origins:
    if type(val) == str:
        val = val.lower()
        if 'leg' in val or 'ltct' in val:
            # print(val)
            val = 'leg'
            # print(val)
        #     if '1' in val or 'pro' in val:
        #         val = 'leg1'
        #     elif '2' in val or 'meso' in val:
        #         val = 'leg2'
        #     elif '3' in val or 'meta' in val:
        #         val = 'leg3'
        #     elif 'tct' in val:
        #         val = 'leg tectulum'
        #     else:
        #         val = 'leg'
        elif 'ntct' in val or 'neck' in val:
            val = 'neck'
        elif 'w' in val:
            val = 'wing'
        elif 'ht' in val or 'ha' in val:
            val = 'haltere'
        elif 'anm' in val or 'ab' in val:
            val = 'abdomen'
        elif 'vac' in val:
            val = 'vac'
        elif 'prothorax' in val:
            val = 'ventral prothorax'
        elif val == 'ov':
            val = 'ov'
        elif 'tct' in val:
            val = 'other tectulum'
        else:
            val = np.nan # brain, notum, cv
    coarse_origin.append(val)
all_meta['origin'] = coarse_origin

In [ ]:
excitatory_mask = (all_meta['top_nt'] == 'acetylcholine').astype(int)
inhibitory_mask = all_meta['top_nt'].isin(('glutamate', 'gaba')).astype(int)
print(inhibitory_mask.sum())

nt_valence = excitatory_mask - inhibitory_mask
all_meta['nt_valence'] = nt_valence

In [ ]:
all_meta['top_nt'].value_counts()

In [ ]:
connective_mask = all_meta['super_class'].isin(('descending', 'ascending', 'sensory_ascending'))
vnc_mask = (all_meta['dataset'] == 'MANC') & ~connective_mask
brain_mask = (all_meta['dataset'] == 'FAFB') & ~connective_mask

In [ ]:
def add_laterality_to_elist(elist):
    pre_side = all_meta.loc[elist['pre'], 'side'].values
    post_side = all_meta.loc[elist['post'], 'side'].values
    
    laterality = np.full(pre_side.shape, 'contralateral')
    laterality[(pre_side == post_side) & (pre_side == 'center')] = 'center_center'
    laterality[(pre_side == post_side) & (pre_side != 'center')] = 'ipsilateral'
    laterality[(pre_side != post_side) & (post_side == 'center')] = 'inward'
    laterality[(pre_side != post_side) & (pre_side == 'center')] = 'outward'

    laterality_num = (laterality == 'ipsilateral').astype(int) - (laterality == 'contralateral').astype(int)
    elist = elist.assign(laterality=laterality, laterality_num=laterality_num)
    return elist

def add_meta_to_elist(elist, columns=[]):
    annotations = {}
    for col in columns:
        for partner in ['pre', 'post']:
            key = f'{partner}_{col}'
            value = all_meta.loc[elist[partner], col].values
            annotations[key] = value
    elist = elist.assign(**annotations)
    
    elist = add_laterality_to_elist(elist)
    
    return elist

def get_elist(conn, source_ids=None, target_ids=None, min_norm=0.):
    if (source_ids is None) and (target_ids is None):
        assert False, 'must provide sources or targets'
    
    conditions = []
    all_ids = []
    if source_ids is not None:
        question_marks = ','.join(['?'] * len(source_ids))
        sub_query = f'(pre IN ({question_marks}))'
        conditions.append(sub_query)
        all_ids = all_ids + source_ids

    if target_ids is not None:
        question_marks = ','.join(['?'] * len(target_ids))
        sub_query = f'(post IN ({question_marks}))'
        conditions.append(sub_query)
        all_ids = all_ids + target_ids

    conditions = ' AND '.join(conditions)

    query = f'SELECT * FROM edgelist_simple WHERE ((norm >= {min_norm}) AND ({conditions}))'
    elist = pd.read_sql_query(query, conn, params=all_ids)
    elist['pre'] = elist['pre'].astype(int)
    elist['post'] = elist['post'].astype(int)

    # remove connections that don't correspond to a neuron in the dataset
    elist = elist[elist['pre'].isin(all_meta.index) & elist['post'].isin(all_meta.index)]
    return elist   

def get_ids(permit=None, col=None):
    if permit is None:
        return None
    mask = all_meta[col].isin(permit) # & all_meta['is_vnc']
    ids = all_meta[mask].index.to_list()
    return ids

def get_source_target_elists(linker, source=None, target=None,
                             linker_col='cell_type', source_col='cell_type', target_col='cell_type',
                             source_min_norm=.01, target_min_norm=.01, exclude_overlap=False):
    linker_ids = get_ids(linker, linker_col)
    source_ids = get_ids(source, source_col)
    target_ids = get_ids(target, target_col)
    # print(linker_ids.min(), linker_ids.max())
    # print(source_ids.min(), source_ids.max())
    # print(target_ids.min(), target_ids.max())

    conn = sqlite3.connect(sql_path)
    source_elist = get_elist(conn, source_ids, linker_ids, source_min_norm)
    target_elist = get_elist(conn, linker_ids, target_ids, target_min_norm)

    if exclude_overlap:
        pre_mask = ~source_elist['pre'].isin(linker_ids)
        if target_ids is not None:
            pre_mask = pre_mask & ~source_elist['pre'].isin(target_ids)
        source_elist = source_elist[pre_mask]
    
        post_mask = ~target_elist['post'].isin(linker_ids)
        if source_ids is not None:
            post_mask = post_mask & ~target_elist['post'].isin(source_ids)
        target_elist = target_elist[post_mask]

        # print(pre_mask.sum() / pre_mask.size, post_mask.sum() / post_mask.size, '!!!')

    extras = ['nt_valence', 'neuromere', 'cell_class', 'super_class', 'origin']
    source_elist = add_meta_to_elist(source_elist, [source_col, linker_col] + extras)
    target_elist = add_meta_to_elist(target_elist, [linker_col, target_col] + extras)

    return source_elist, target_elist

def elist_to_grouped_matrix(elist, pre_col, post_col, pre_vals=None, post_vals=None,
                            weight_col='count', laterality_annotation=None, simplify_column_names=True):
    pre_key = f'pre_{pre_col}'
    post_key = f'post_{post_col}'
    
    if laterality_annotation is not None:
        laterality = elist['laterality']
        key = pre_key if laterality_annotation == 'pre' else post_key
        values = elist[key]
        elist[key] = elist[key] + '_' + laterality

        # print(elist[key])

    elist = elist.groupby([pre_key, post_key], as_index=False).agg(weight=(weight_col, 'sum'))
    matrix = elist.pivot(index=pre_key, columns=post_key, values='weight')
    if simplify_column_names:
        matrix = matrix.rename(index={pre_key: 'pre'}, columns={post_key: 'post'})
        # print('Renamed!!!!!')
    if pre_vals is None:
        pre_vals = sorted(matrix.index.to_list())
    if post_vals is None:
        post_vals = sorted(matrix.columns.to_list())
    matrix = matrix.reindex(index=pre_vals, columns=post_vals).fillna(0)
    return matrix

def get_source_target_grouped_matrices(linker, source=None, target=None,
                                       linker_col='cell_type', source_col='cell_type', target_col='cell_type',
                                       lateralize=True, source_min_norm=.0, target_min_norm=.0,
                                       exclude_overlap=True, exclude_empty=False, return_elist=True, aggregated_min_norm=0):
    source_elist, target_elist = get_source_target_elists(linker, source, target,
                                                          linker_col, source_col, target_col,
                                                          source_min_norm, target_min_norm, exclude_overlap)

    # TODO! make elist_to_grouped_matrix work with user specified source target. i.e. modify post vals with laterality
    # source_matrix = elist_to_grouped_matrix(source_elist, source_col, linker_col, source, linker, laterality_annotation='pre')
    # target_matrix = elist_to_grouped_matrix(target_elist, linker_col, target_col, linker, target, laterality_annotation='post')

    if lateralize:
        source_annotation = 'pre'
        target_annotation = 'post'
    else:
        source_annotation = target_annotation = None
    source_matrix = elist_to_grouped_matrix(source_elist, source_col, linker_col, None, linker,
                                            laterality_annotation=source_annotation)
    target_matrix = elist_to_grouped_matrix(target_elist, linker_col, target_col, linker, None,
                                            laterality_annotation=target_annotation)

    if exclude_empty:
        mask = (source_matrix.values.sum(0) > 0) & (target_matrix.values.sum(1) > 0)
        source_matrix = source_matrix.iloc[:, mask]
        target_matrix = target_matrix.iloc[mask]

        mask = source_matrix.values.sum(1) > 0
        source_matrix = source_matrix.iloc[mask]
        
        mask = target_matrix.values.sum(0) > 0
        target_matrix = target_matrix.iloc[:, mask]

    if return_elist:
        return source_matrix, target_matrix, source_elist, target_elist
    else:
        return source_matrix, target_matrix

In [ ]:
def filter_by_norm(matrix, min_norm=0):
    normalized = matrix.values / matrix.values.sum(-1, keepdims=True)
    mask = np.any(normalized >= min_norm, axis=0)
    matrix = matrix.iloc[:, mask]
    return matrix

## Load Connectivity

In [ ]:
super_classes = ['descending']#, 'ascending', 'sensory_ascending']

linker_col = 'cell_type'
source_col = 'cell_type'
target_col = 'cell_type' # 'cell_type'

brain_types = np.unique(all_meta[brain_mask & ~all_meta[source_col].isnull()][source_col].values)
vnc_types = np.unique(all_meta[vnc_mask & ~all_meta[target_col].isnull()][target_col].values)

source_target_types = [(brain_types, vnc_types), (vnc_types, brain_types), (vnc_types, brain_types)]

source_matrices = {}
target_matrices = {}
elists = {}

profiles = {}
labels = {}
labels_good = {}
min_norm = 0 #.01
aggregated_min_norm = 0.05
exclude_overlap = True
exclude_empty = True
lateralize = True

for super_class, (source_types, target_types) in zip(super_classes, source_target_types):
    print(super_class)
    linker_types = set(all_meta[all_meta['super_class'] == super_class][linker_col].values)
    
    source_matrix, target_matrix, source_elist, target_elist = \
            get_source_target_grouped_matrices(linker_types, source_types, target_types,
                                               linker_col, source_col, target_col,
                                               lateralize=lateralize,
                                               source_min_norm=min_norm, target_min_norm=min_norm,
                                               exclude_overlap=exclude_overlap, exclude_empty=exclude_empty)
    print(source_matrix.shape, target_matrix.shape, 'pre filter')
    
    source_matrix = filter_by_norm(source_matrix.transpose(), aggregated_min_norm).transpose()
    target_matrix = filter_by_norm(target_matrix, aggregated_min_norm)
    print(source_matrix.shape, target_matrix.shape, 'post filter')

    source_matrices[super_class] = source_matrix
    target_matrices[super_class] = target_matrix

    unnormed_profiles = {
        'input': source_matrix.values.T,
        'output': target_matrix.values
    }
    for k, v in unnormed_profiles.items():
        # profiles[f'{super_class}_{k}'] = v / v.sum(-1, keepdims=True)
        profiles[f'{super_class}_{k}'] = v / np.linalg.norm(v, axis=-1, keepdims=True)
        labels_good[f'{super_class}_{k}'] = source_matrix.index.values if k == 'input' else target_matrix.columns.values
        elists[f'{super_class}_{k}'] = source_elist if k == 'input' else target_elist

    labels[super_class] = [source_matrix.columns.values, source_matrix.index.values, target_matrix.columns.values]

    # profiles[f'{super_class}_both'] = np.concatenate((profiles[f'{super_class}_input'], profiles[f'{super_class}_output']), axis=-1)
    # profiles[f'{super_class}_both'] = profiles[f'{super_class}_both'] / profiles[f'{super_class}_both'].sum(-1, keepdims=True)

print(profiles.keys())

In [ ]:
label_types = {}
label_sides = {}
for k, v in labels_good.items():
    sides = []
    cell_types = []
    for token in v:
        cell_type, side_name = token.rsplit('_', 1)
        if side_name.endswith('ipsilateral'):
            side = 1
        elif side_name.endswith('contralateral'):
            side = -1
        else:
            side = 0
        sides.append(side)
        cell_types.append(cell_type)
    label_sides[k] = np.array(sides)
    label_types[k] = np.array(cell_types)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

for k in profiles:
    cos_sims = cosine_similarity(profiles[k], profiles[k])
    np.fill_diagonal(cos_sims, 0)
    plt.imshow(cos_sims)
    plt.title(k)
    plt.show()

In [ ]:
cos_similarities = {}
for k, v in profiles.items():
    cos_similarities[k] = cosine_similarity(profiles[k], profiles[k])

for super_class in super_classes:
    input_sims = np.triu(cos_similarities[f'{super_class}_input'], 1)
    output_sims = np.triu(cos_similarities[f'{super_class}_output'], 1)
    plt.scatter(input_sims, output_sims, s=1, alpha=.5)
    plt.title(super_class)
    plt.xlabel('input similarity')
    plt.ylabel('output similarity')
    plt.xlim(-.01, 1.01)
    plt.ylim(-.01, 1.01)
    plt.gca().set_aspect('equal')
    plt.show()

In [ ]:
# for k, prof in profiles.items():
#     print(k)
#     print(prof.shape)
#     for i, p in enumerate(prof):
#         plt.plot(np.sort(p))
#         plt.title(i)
#         plt.show()

In [ ]:
# # simplify by focusing on output
for k in super_classes:
    if del_input:
        del profiles[f'{k}_input']
    # # simplify by focusing on output
    if del_output:
        del profiles[f'{k}_output']

In [ ]:
# cos_sim = cosine_similarity(profiles['descending_output'].T, profiles['descending_output'].T)
# plt.imshow(cos_sim)
# plt.xticks(range(len(cos_sim)), labels['descending'][2], rotation=90)
# plt.yticks(range(len(cos_sim)), labels['descending'][2])
# plt.colorbar()
# plt.show()

## Perform Embeddings

### Principle Component Analysis

In [ ]:
def plot_explained_variance(pca, title=''):
    # Compute cumulative and per-component explained variance
    cumulative_variance = pca.explained_variance_ratio_.cumsum()
    explained_variance = pca.explained_variance_ratio_
    
    fig, ax1 = plt.subplots(figsize=(8, 5))
    
    # Plot the per-component explained variance as bars
    ax1.bar(range(len(explained_variance)), explained_variance, alpha=0.6, color='g')
    ax1.set_xlabel('Number of Components')
    ax1.set_ylabel('Explained Variance per Component', color='g')
    
    # Set y-axis color and ticks for the bar plot
    ax1.tick_params(axis='y', labelcolor='g')
    ax1.set_ylim(0, 1.1 * explained_variance.max())
    
    # Create a second y-axis to plot cumulative explained variance
    ax2 = ax1.twinx()
    ax2.plot(range(len(cumulative_variance)), cumulative_variance, color='b')
    ax2.set_ylabel('Cumulative Explained Variance', color='b')

    index = np.flatnonzero(cumulative_variance >= .9).min()
    ax2.vlines(index, 0, 1.1, linestyle='dotted', color='black')
    
    # Set y-axis color and ticks for the cumulative explained variance plot
    ax2.tick_params(axis='y', labelcolor='b')
    ax2.set_ylim(0, 1.1)
    
    # # Add a grid for better visualization
    # ax1.grid(True)
    
    # Add title
    plt.title(title)

    # Show the plot
    plt.show()

In [ ]:
from sklearn.decomposition import PCA

pca_models = {}
pca_projections = {}
for k, v in profiles.items():
    # # It's not clear that we should normalize
    # v = v - v.mean(0)
    # v = v / np.std(v, 0)
    
    pca_models[k] = PCA().fit(v)
    pca_projections[k] = pca_models[k].transform(v)
    plot_explained_variance(pca_models[k], k)

In [ ]:
for k, pca_model in pca_models.items():
    super_class, in_out = k.split('_')
    in_out = 1 if in_out == 'input' else 2
    model_labels = labels[super_class][in_out]
    # print(len(model_labels), pca_model.components_.shape)
    for i in range(10):
        pc_i = pca_model.components_[i]
        argmax = np.argsort(-np.abs(pc_i))[:10]
        for j in argmax:
            print(j, model_labels[j], pc_i[j])
        plt.plot(pc_i)
        plt.show()

In [ ]:
# for k, proj in pca_projections.items():
#     super_class, in_out = k.split('_')
#     in_out = 1 if in_out == 'input' else 2
#     model_labels = labels[super_class][0]
#     # print(len(model_labels), pca_model.components_.shape)
#     for i in range(10):
#         argmax = np.argsort(-np.abs(proj[:, i]))[:10]
#         for j in argmax:
#             print(j, model_labels[j], proj[j, i])
#         plt.plot(proj[:, i])
#         plt.show()

In [ ]:
# for k, proj in pca_projections.items():
#     plt.scatter(*proj[:, (0, 1)].T)
#     plt.title(k)
#     plt.show()
#     plt.scatter(*proj[:, (0, 2)].T)
#     plt.title(k)
#     plt.show()
#     plt.scatter(*proj[:, (1, 2)].T)
#     plt.title(k)
#     plt.show()

### UMAP embedding

In [ ]:
import umap

def perform_umap_embedding(data, n_neighbors=10, min_dist=0.1, n_components=2, random_state=None, metric='cosine'):
    """
    Perform UMAP embedding on the input data matrix.

    Parameters:
    - data: np.ndarray or pd.DataFrame
        Input data matrix of shape (num_samples, sample_dimension).
    - n_neighbors: int, optional (default=15)
        The size of the local neighborhood used for manifold approximation.
    - min_dist: float, optional (default=0.1)
        The minimum distance between points in the low-dimensional space.
    - n_components: int, optional (default=2)
        The dimension of the low-dimensional embedding (typically 2 or 3).
    - random_state: int or None, optional (default=None)
        Random seed for reproducibility.

    Returns:
    - embedding: np.ndarray
        UMAP-embedded data of shape (num_samples, n_components).
    """
    # Initialize UMAP model with parameters
    umap_model = umap.UMAP(n_neighbors=n_neighbors, 
                           min_dist=min_dist, 
                           n_components=n_components, 
                           random_state=random_state,
                           metric=metric)
    
    # Perform the embedding
    embedding = umap_model.fit_transform(data)
    
    return umap_model, embedding

In [ ]:
umap_models = {}
umap_embeddings = {}

n_neighbors = 15 #15
min_dist = 0
random_state = 4

for k, v in profiles.items():
    np.random.seed(random_state)
    random.seed(random_state)
    umap_models[k], umap_embeddings[k] = perform_umap_embedding(v, min_dist=min_dist, n_neighbors=n_neighbors, random_state=random_state)
    # umap_models[k], umap_embeddings[k] = perform_umap_embedding(pca_projections[k][:, :20], min_dist=min_dist, n_neighbors=n_neighbors)

    plt.scatter(*umap_embeddings[k].T)
    plt.title(k + ' umap')
    plt.show()

In [ ]:
from sklearn.decomposition import PCA

pca_models = {}
pca_projections = {}
for k, v in profiles.items():
    # # It's not clear that we should normalize
    # v = v - v.mean(0)
    # v = v / np.std(v, 0)
    
    pca_models[k] = PCA().fit(v)
    pca_projections[k] = pca_models[k].transform(v)
    plot_explained_variance(pca_models[k], k)

In [ ]:
# for k in profiles:
#     embedding = umap_embeddings[k]
#     for i in range(10):
#         projection = pca_projections[k][:, i]
#         color = assign_colors(projection)
#         plt.scatter(*umap_embeddings[k].T, c=color)
#         plt.title(f'{k}_{i}')
#         plt.show()
#     plt.show()

In [ ]:
# for k, pca_model in pca_models.items():
#     print(k)
#     super_class, in_out = k.split('_')
#     in_out = 1 if in_out == 'input' else 2
#     model_labels = labels[super_class][in_out]
#     # print(len(model_labels), pca_model.components_.shape)
#     for i in range(10):
#         pc_i = pca_model.components_[i]
#         argmax = np.argsort(-np.abs(pc_i))[:10]
#         for j in argmax:
#             print(j, model_labels[j], pc_i[j])
#         plt.plot(pc_i)
#         plt.show()

### NMF

In [ ]:
from sklearn.decomposition import NMF
n_nmf_factors = 16
random_state = 0
nmf_models = {}
nmf_projections = {}
for k, v in profiles.items():
    # # It's not clear that we should normalize
    # v = v - v.mean(0)
    # v = v / np.std(v, 0)

    np.random.seed(random_state)
    random.seed(random_state)
    nmf_models[k] = NMF(n_nmf_factors, random_state=random_state).fit(v)
    nmf_projections[k] = nmf_models[k].transform(v)
    embedding = umap_embeddings[k]
    # for i in range(n_nmf_factors):
    #     projection = nmf_projections[k][:, i]
    #     color = assign_colors(projection)
    #     plt.scatter(*umap_embeddings[k].T, c=color)
    #     plt.title(f'{k}_{i}')
    #     plt.show()

In [ ]:
for k, nmf_model in nmf_models.items():
    print(k)
    super_class, in_out = k.split('_')
    in_out = 1 if in_out == 'input' else 2
    model_labels = labels[super_class][in_out]
    # print(len(model_labels), pca_model.components_.shape)
    for i in range(10):
        pc_i = nmf_model.components_[i]
        argmax = np.argsort(-np.abs(pc_i))[:10]
        for j in argmax:
            print(j, model_labels[j], pc_i[j])
        plt.plot(pc_i)
        plt.show()

In [ ]:
# from sklearn.decomposition import NMF
# import tqdm
# for k, v in profiles.items():
#     # # It's not clear that we should normalize
#     # v = v - v.mean(0)
#     # v = v / np.std(v, 0)
#     recon = []
#     for n_nmf_factors in tqdm.trange(1, 100, 5):
#         err = NMF(n_nmf_factors, random_state=random_state, max_iter=1000).fit(v > 0).reconstruction_err_
#         recon.append(err)
#     plt.plot(recon)
#     plt.show()
#     # for i in range(n_nmf_factors):
#     #     projection = nmf_projections[k][:, i]
#     #     color = assign_colors(projection)
#     #     plt.scatter(*umap_embeddings[k].T, c=color)
#     #     plt.title(f'{k}_{i}')
#     #     plt.show()
# # nmf_models['descending_input'].reconstruction_err_

### Spectral Clustering

In [ ]:
from sklearn.cluster import SpectralClustering

def perform_spectral_clustering(data, n_clusters=2, n_components=None, affinity='rbf', random_state=None):
    """
    Perform spectral clustering on the input data matrix.

    Parameters:
    - data: np.ndarray or pd.DataFrame
        Input data matrix of shape (num_samples, sample_dimension).
    - n_clusters: int, optional (default=2)
        The number of clusters to form.
    - affinity: str, optional (default='rbf')
        How to construct the affinity matrix. Options:
        - 'rbf': uses a radial basis function (Gaussian) kernel.
        - 'nearest_neighbors': uses k-nearest neighbors to construct the affinity matrix.
        - 'precomputed': a precomputed affinity matrix can be passed instead of raw data.
    - random_state: int or None, optional (default=None)
        Random seed for reproducibility.

    Returns:
    - labels: np.ndarray
        Array of cluster labels for each data point.
    """
    # Initialize SpectralClustering model with parameters
    clustering_model = SpectralClustering(n_clusters=n_clusters, 
                                          affinity='precomputed', 
                                          random_state=random_state,
                                          n_components=n_components)

    # Fit the model and predict the cluster labels
    data = data / np.linalg.norm(data, -1, keepdims=True)
    cosine_sim_matrix = data @ data.T
    labels = clustering_model.fit_predict(cosine_sim_matrix)
    
    return clustering_model, labels

In [ ]:
clustering_models = {}
cluster_assignments = {}
n_clusters = {'descending_output': 6, 'descending_input': 14, 'ascending_input': 10}
random_state = 0
np.random.seed(random_state)
random.seed(random_state)

cmap = plt.get_cmap('tab20', 20)


# cluster_remapping = {
#     'descending_output': {
#         5: 0,
#         4: 1,
#         1: 2,
#         0: 3,
#         2: 4,
#         3: 5
#     }
# }
cluster_remapping = {}
for k, v in profiles.items():
    # v = v[:, :100]
    clustering_models[k], cluster_assignments[k] = perform_spectral_clustering(v, n_clusters=n_clusters[k], random_state=random_state)

    # c = cmap(cluster_assignments[k])

    if k in cluster_remapping:
        cluster_assignments[k] = np.array([cluster_remapping[k][i] for i in cluster_assignments[k]])

    
    for i in np.unique(cluster_assignments[k]):
        print(i)
        mask = cluster_assignments[k] == i
        plt.scatter(*umap_embeddings[k][mask].T, label=i) #, c=cmap(i))
    plt.title(k + ' umap')
    plt.legend()
    plt.show()

In [ ]:
for k, v in cluster_assignments.items():
    cos_sims = profiles[k] @ profiles[k].T
    np.fill_diagonal(cos_sims, 0)
    # plt.imshow(cos_sims)
    # plt.title(k + ' unsorted')
    # plt.show()
    reordering = np.argsort(cluster_assignments[k])
    reordered_assignments = cluster_assignments[k][reordering]
    boundaries = np.flatnonzero(reordered_assignments[1:] != reordered_assignments[:-1]) + .5
    cos_sims = cos_sims[reordering][:, reordering]
    
    plt.imshow(cos_sims, interpolation='none')
    plt.title(k + ' spectral')
    plt.show()

In [ ]:
# for k, v in cluster_assignments.items():
#     cos_sims = profiles[k] @ profiles[k].T
#     np.fill_diagonal(cos_sims, 0)
#     # plt.imshow(cos_sims)
#     # plt.title(k + ' unsorted')
#     # plt.show()
#     reordering = np.argsort(cluster_assignments[k])
#     reordered_assignments = cluster_assignments[k][reordering]
#     boundaries = np.flatnonzero(reordered_assignments[1:] != reordered_assignments[:-1]) + .5
#     cos_sims = cos_sims[reordering][:, reordering]
#     plt.imshow(cos_sims)
#     plt.vlines(boundaries, 0, len(cos_sims) - .5, color='r', linestyle='dotted')
#     plt.hlines(boundaries, 0, len(cos_sims) - .5, color='r', linestyle='dotted')
#     plt.title(k + ' spectral')
#     plt.show()

## Aggregate Metadata

In [ ]:
def get_weighted_average(elist, col, direction):
    key = f'{direction}_{linker_col}'
    total_out = elist.loc[:, (key, 'count')].groupby(key).sum().loc[linker_types]
    elist = elist.assign(**{f'{col}_temp': elist['count'] * elist[col]})
    weighted_val = elist.loc[:, (key, f'{col}_temp')].groupby(key).sum().loc[linker_types]
    return weighted_val.values / total_out.values



In [ ]:
def elist_to_grouped_matrix(elist, pre_col, post_col, pre_vals=None, post_vals=None,
                            weight_col='count', laterality_annotation=None, simplify_column_names=True):
    pre_key = f'pre_{pre_col}'
    post_key = f'post_{post_col}'
    
    if laterality_annotation is not None:
        laterality = elist['laterality']
        key = pre_key if laterality_annotation == 'pre' else post_key
        values = elist[key]
        elist[key] = elist[key] + '_' + laterality

        # print(elist[key])

    elist = elist.groupby([pre_key, post_key], as_index=False).agg(weight=(weight_col, 'sum'))
    matrix = elist.pivot(index=pre_key, columns=post_key, values='weight')
    if simplify_column_names:
        matrix = matrix.rename(index={pre_key: 'pre'}, columns={post_key: 'post'})
        # print('Renamed!!!!!')
    if pre_vals is None:
        pre_vals = sorted(matrix.index.to_list())
    if post_vals is None:
        post_vals = sorted(matrix.columns.to_list())
    matrix = matrix.reindex(index=pre_vals, columns=post_vals).fillna(0)
    return matrix

In [ ]:
laterality_annotation = None

annotated_meta = {}
cluster_meta = {}

for k in profiles:
    super_class, direction = k.split('_')
    linker_types = source_matrices[super_class].columns.values

    elist = elists[k]
    
    mask = all_meta[linker_col].isin(linker_types)
    sub_meta = all_meta[mask].set_index(linker_col, drop=False)
    sub_meta = sub_meta[~sub_meta.index.duplicated(keep='first')]
    sub_meta = sub_meta.loc[linker_types, ('id', 'nt_valence')]


    sub_meta['cluster'] = cluster_assignments[k]
        
    # dn_letters = np.array([x[2] for x in linker_types])
    # sub_meta['DN_letter'] = dn_letters

    type_to_valence = all_meta.loc[:, ('cell_type', 'nt_valence')].groupby(linker_col).mean()
    nt_valences = type_to_valence.loc[linker_types]

    for col in ['laterality_num', 'post_nt_valence']:
        sub_meta[col] = get_weighted_average(elist, col, direction='pre' if 'output' in k else 'post')

    for col in ['origin', 'super_class']:
        if direction == 'output':
            grouped_matrix = elist_to_grouped_matrix(elist, linker_col, col, linker_types, None, laterality_annotation=laterality_annotation)
        else:
            grouped_matrix = elist_to_grouped_matrix(elist, col, linker_col, None, linker_types, laterality_annotation=laterality_annotation)
            grouped_matrix = grouped_matrix.transpose()

        grouped_matrix = grouped_matrix.div(grouped_matrix.sum(axis=1), axis=0)
        # grouped_matrix = grouped_matrix.div(grouped_matrix.sum(axis=0), axis=1)

        new_cols = {f'{col}_{val}_prop': grouped_matrix.loc[:, val] for val in grouped_matrix.columns}
        sub_meta = sub_meta.assign(**new_cols).copy()

    annotated_meta[k] = sub_meta
    cluster_meta[k] = sub_meta.groupby('cluster').mean()

## Visualization

In [ ]:
cluster_meta[k]

In [ ]:
for k in cluster_meta:
    print(k)
    for col in cluster_meta[k].columns:
        print(col)
        print(cluster_meta[k][col])
        print()

In [ ]:
def assign_colors(array, cmap_name='plasma', symmetrize=False):
    # Normalize the array between 0 and 1
    if symmetrize:
        norm = array / np.abs(array).max()
        norm = (norm + 1) / 2 
    else:
        norm = (array - np.min(array)) / (np.max(array) - np.min(array))
    
    # Get the colormap
    cmap = cm.get_cmap(cmap_name)
    
    # Apply colormap to normalized values
    colors = cmap(norm)
    
    return colors

In [ ]:
for k, v in cluster_assignments.items():
    cos_sims = cos_similarities[k]
    np.fill_diagonal(cos_sims, 0)
    # plt.imshow(cos_sims)
    # plt.title(k + ' unsorted')
    # plt.show()
    reordering = np.argsort(cluster_assignments[k])
    reordered_assignments = cluster_assignments[k][reordering]
    boundaries = np.flatnonzero(reordered_assignments[1:] != reordered_assignments[:-1]) + .5
    cos_sims = cos_sims[reordering][:, reordering]
    
    plt.imshow(cos_sims, interpolation='none')
    plt.title(k + ' spectral')
    plt.show()

    plt.imshow(cos_sims, interpolation='none', vmax=.25)
    plt.title(k + ' spectral thresholded')
    plt.show()

In [ ]:
for k in profiles:
    sub_meta = annotated_meta[k]
    for col in ['cluster']:# , 'DN_letter']:
        embedding = umap_embeddings[k]
        for i in np.sort(sub_meta[col].unique()):
            mask = sub_meta[col] == i
            plt.scatter(*umap_embeddings[k][mask].T, label=i)
        plt.title(col)
        plt.legend()
        plt.show()
        # plt.show()

In [ ]:
# for dn in annotated_meta['descending_output'][annotated_meta['descending_output']['cluster'] == 1].index:
#     print(dn[2])

In [ ]:
for k in profiles:
    sub_meta = annotated_meta[k]
    for col in ['cluster']: #, 'DN_letter']:
        embedding = umap_embeddings[k]
        for i in np.sort(sub_meta[col].unique()):
            mask = sub_meta[col] == i
            plt.scatter(*umap_embeddings[k].T, c='black')
            plt.scatter(*umap_embeddings[k][mask].T, c='red')
            plt.title(f'{col}_{i}')
            plt.show()
        # plt.show()

In [ ]:
for k in profiles:
    sub_meta = annotated_meta[k]

    for col in ['laterality_num', 'nt_valence', 'post_nt_valence']:
        embedding = umap_embeddings[k]
        score = sub_meta[col]
        color = assign_colors(score, 'coolwarm', True)
        plt.scatter(*umap_embeddings[k].T, c=color)
        plt.title(col)
        plt.show()
        # plt.show()

In [ ]:
num_comps = 12
for k in profiles:
    sub_meta = annotated_meta[k]
    for algo, proj in {'pca': pca_projections[k], 'nmf': nmf_projections[k]}.items():
        embedding = umap_embeddings[k]
        for i in range(min(num_comps, proj.shape[-1])):
            score = proj[:, i]
            color = assign_colors(score)
            plt.scatter(*umap_embeddings[k].T, c=color)
            plt.title(f'{k} {algo} {i}')
            plt.show()
            # plt.show()

In [ ]:
num_comps = 6
for k in profiles:
    sub_meta = annotated_meta[k]
    for col in np.sort([col for col in sub_meta.columns if col.endswith('prop')]):
        embedding = umap_embeddings[k]
        score = sub_meta[col]
        color = assign_colors(score)
        mask = score > 0.001
        # plt.scatter(*umap_embeddings[k].T, c='black', alpha=.1)
        plt.scatter(*umap_embeddings[k].T, c=color, alpha=1)
        plt.scatter(*umap_embeddings[k][mask].T, c=color[mask], alpha=1)
        plt.title(col)
        plt.show()
            # plt.show()

## Overlaying Output Metadata onto Input Space

In [ ]:
data_key = 'descending_output'
# data_key = 'descending_input'

for k in profiles:
    if k == data_key:
        continue
    sub_meta = annotated_meta[data_key]

    for col in ['laterality_num', 'nt_valence', 'post_nt_valence']:
        embedding = umap_embeddings[k]
        score = sub_meta[col]
        color = assign_colors(score, 'coolwarm', True)
        plt.scatter(*umap_embeddings[k].T, c=color)
        plt.title(col)
        plt.show()
        # plt.show()

    for col in ['laterality_num', 'nt_valence', 'post_nt_valence']:
        embedding = umap_embeddings[k]
        score = np.abs(sub_meta[col])
        color = assign_colors(score, 'plasma', False)
        plt.scatter(*umap_embeddings[k].T, c=color)
        plt.title(col)
        plt.show()
        # plt.show()

for k, v in cluster_assignments.items():
    if k == data_key:
        continue
    cos_sims = cos_similarities[k]
    np.fill_diagonal(cos_sims, 0)
    # plt.imshow(cos_sims)
    # plt.title(k + ' unsorted')
    # plt.show()
    reordering = np.argsort(cluster_assignments[data_key])
    reordered_assignments = cluster_assignments[k][reordering]
    boundaries = np.flatnonzero(reordered_assignments[1:] != reordered_assignments[:-1]) + .5
    cos_sims = cos_sims[reordering][:, reordering]
    
    plt.imshow(cos_sims, interpolation='none')
    plt.title(k + ' spectral')
    plt.show()

    plt.imshow(cos_sims, interpolation='none', vmax=.25)
    plt.title(k + ' spectral thresholded')
    plt.show()

In [ ]:
for k in profiles:
    if k == data_key:
        continue
    sub_meta = annotated_meta[data_key]
    for col in ['cluster']:# , 'DN_letter']:
        embedding = umap_embeddings[k]
        for i in np.sort(sub_meta[col].unique()):
            mask = sub_meta[col] == i
            plt.scatter(*umap_embeddings[k][mask].T, label=i)
        plt.title(col)
        plt.legend()
        plt.show()
        # plt.show()

In [ ]:
list(labels_good['descending_input']).index('BM_Taste_ipsilateral'), list(labels_good['descending_input']).index('BM_Taste_contralateral')

In [ ]:
# profiles['descending_output'].sum(-1)

In [ ]:
taste_input = profiles['descending_input'][:, 32:34].sum(-1)
for i in np.argsort(taste_input)[::-1]:
    print(labels['descending'][0][i], taste_input[i])

In [ ]:
num_comps = 6
for k in profiles:
    if k == data_key:
        continue
    sub_meta = annotated_meta[data_key]
    for col in np.sort([col for col in sub_meta.columns if col.endswith('prop')]):
        embedding = umap_embeddings[k]
        score = sub_meta[col]
        color = assign_colors(score)
        mask = score > 0.001
        # plt.scatter(*umap_embeddings[k].T, c='black', alpha=.1)
        plt.scatter(*umap_embeddings[k].T, c=color, alpha=1)
        plt.scatter(*umap_embeddings[k][mask].T, c=color[mask], alpha=1)
        plt.title(col)
        plt.show()
            # plt.show()

In [ ]:
num_comps = 12
for k in profiles:
    if k == data_key:
        continue
    sub_meta = annotated_meta[data_key]
    for algo, proj in {'nmf': nmf_projections[data_key]}.items():
        embedding = umap_embeddings[k]
        for i in range(min(num_comps, proj.shape[-1])):
            score = proj[:, i]
            color = assign_colors(score)
            plt.scatter(*umap_embeddings[k].T, c=color, alpha=.5)
            mask = score > .5 * score.max()
            plt.scatter(*umap_embeddings[k][mask].T, c=color[mask])
            plt.title(f'{data_key} {algo} {i}')
            plt.xlabel(f'{k}_umap_0')
            plt.ylabel(f'{k}_umap_1')
            plt.show()
            # plt.show()

In [ ]:
for k in cluster_meta:
    for col in cluster_meta[k].columns:
        print(col)
        print(cluster_meta[k][col])
        print()

In [ ]:
annotated_meta[k].columns

In [ ]:
for k in profiles:
    sub_meta = annotated_meta[k]
    for col in ['cluster']:# , 'DN_letter']:
        xcol = 'laterality_num'
        ycol = 'super_class_dorsal_prop'
        embedding = np.stack((annotated_meta['descending_output'][xcol], annotated_meta['descending_output'][ycol]), axis=-1)
        print(embedding.shape)
        for i in np.sort(sub_meta[col].unique()):
            mask = sub_meta[col] == i
            plt.scatter(*embedding[mask].T, label=i)
        plt.title(col)
        plt.xlabel(xcol)
        plt.ylabel(ycol)
        plt.legend()
        plt.show()
        # plt.show()

## Other

In [ ]:
for k, nmf_model in nmf_models.items():
    embedding = umap_embeddings[k]
    
    print(k)
    super_class, in_out = k.split('_')
    in_out = 1 if in_out == 'input' else 2
    model_labels = labels[super_class][in_out]
    # print(len(model_labels), pca_model.components_.shape)
    for i in range(12):
        score = nmf_projections[k][:, i]
        color = assign_colors(score)

        plt.scatter(*umap_embeddings[k].T, c=color)
        plt.title(f'nmf {i}')
        plt.show()

        pc_i = nmf_model.components_[i]
        argmax = np.argsort(-np.abs(pc_i))[:10]
        for j in argmax:
            print(j, model_labels[j], pc_i[j])
        plt.plot(pc_i)
        plt.show()

        argmax = np.argsort(-np.abs(score))[:10]
        for j in argmax:
            print(j, labels[super_class][0][j], score[j])
        plt.plot(score)
        plt.show()

        print('\n' * 50)

In [ ]:
for ind, (k, v) in enumerate(cluster_assignments.items()):
    print(k)
    cos_sims = profiles[k] @ profiles[k].T
    np.fill_diagonal(cos_sims, 0)
    # plt.imshow(cos_sims)
    # plt.title(k + ' unsorted')
    # plt.show()
    for i in np.unique(v):
        mask = (v == i)
        count = mask.sum()
        if count < 10:
            continue
        mean_profile = profiles[k][mask].mean(0)
        mean_profile = mean_profile / mean_profile.sum(0)
        plt.plot(mean_profile * (mean_profile >= .005))

        print('cluster', i)
        print(count)
        biggest_contributors = np.argsort(mean_profile)[::-1]
        for j in biggest_contributors[:10]:
            print(mean_profile[j], labels[k.split('_')[0]][1 if 'input' in k else 2][j])
        print()

    
    plt.title(k)
    plt.show()


In [ ]:
for ind, (k, v) in enumerate(cluster_assignments.items()):
    print(k)
    cos_sims = profiles[k] @ profiles[k].T
    np.fill_diagonal(cos_sims, 0)
    
    for i in np.unique(v):
        mask = (v == i)
        count = mask.sum()
        if count < 10:
            continue
        mean_profile = profiles[k][mask].mean(0)
        mean_profile = mean_profile / mean_profile.sum(0)

        s = mean_profile @ label_sides[k]
        print(i, s, (s + 1) / 2)

In [ ]:
for ind, (k, v) in enumerate(cluster_assignments.items()):
    print(k)
    cos_sims = profiles[k] @ profiles[k].T
    np.fill_diagonal(cos_sims, 0)
    
    for i in np.unique(v):
        mask = (v == i)
        count = mask.sum()
        if count < 10:
            continue
        mean_profile = profiles[k][mask].mean(0)
        mean_profile = mean_profile / mean_profile.sum(0)

        s = mean_profile @ label_sides[k]
        print(i, s, (s + 1) / 2)

In [ ]:
stat_cols = ['top_nt', 'neuromere', 'cell_class', 'super_class', 'origin']

for k, v in cluster_assignments.items():
    print(k)
    for stat_col in stat_cols:
        print(stat_col)
        col = source_col if k.endswith('input') else target_col
        for i in np.unique(v):
            mask = (v == i)
            count = mask.sum()
            if count < 10:
                continue
            mean_profile = profiles[k][mask].mean(0)
            mean_profile = mean_profile / mean_profile.sum(0)
            inds = np.argsort(mean_profile * -1)[:len(mean_profile) // 10]
    
            member_types = label_types[k][inds]
            cluster_meta = all_meta[all_meta[col].isin(member_types)]
            print(i)
            print(cluster_meta[stat_col].value_counts())
        print('-' * 100)

In [ ]:
all_meta['origin'].isnull().sum(), len(all_meta)

In [ ]:
all_meta[all_meta['cell_class'] == 'vnc_motor_neuron']['origin'].value_counts()

In [ ]:
all_meta['neuromere'].value_counts()

In [ ]:
def calc_weighted_sum(value_counts):
    ks = np.array(value_counts.keys())
    vs = value_counts.values
    vs = vs / vs.sum()
    return ks @ vs

In [ ]:
stat_cols = ['nt_valence']

for k, v in cluster_assignments.items():
    print(k)
    for stat_col in stat_cols:
        print(stat_col)
        col = source_col if k.endswith('input') else target_col
        for i in np.unique(v):
            mask = (v == i)
            count = mask.sum()
            if count < 10:
                continue
            mean_profile = profiles[k][mask].mean(0)
            mean_profile = mean_profile / mean_profile.sum(0)
            inds = np.argsort(mean_profile * -1)[:len(mean_profile) // 10]
    
            member_types = label_types[k][inds]
            cluster_meta = all_meta[all_meta[col].isin(member_types)]
            print(i)
            value_counts = cluster_meta[stat_col].value_counts()
            score = calc_weighted_sum(value_counts)
            print(score, (score + 1) / 2)
            

            
        print('-' * 100)

### Visualize Skeletons

In [ ]:
# from fafbseg import flywire
# NC = flywire.NeuronCriteria
# # flywire.set_default_dataset('flat_630')
# flywire.set_default_dataset('public')

# skeletons = {}
# flywire.get_skeletons(NC(super_class='descending'))

In [ ]:
# for k, table in sub_meta.items():
#     print(k)